In [4]:
# %% [colab] BUSI → MedSAM fine-tune (box-prompted SAM on ultrasound)
# ===========================================================
# Train a medical SAM (ViT-B) checkpoint to output lesion masks
# given a bounding-box prompt derived from each GT mask.
# Includes: memory-friendly settings, grad-accumulation, early stopping.
# ===========================================================

# ---------- MEMORY/ENV ----------
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
# optional: uncomment to be even stricter
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# ---------- INSTALL ----------
!pip -q install --no-input "transformers==4.53.3" "accelerate>=0.33" kagglehub albumentations opencv-python pillow timm==0.9.16

# ---------- CONFIG ----------
DATA_DIR = None                     # None → auto-download BUSI via kagglehub; or set to your local path
RUNS_DIR = "/content/runs_busi_medsam_box"
INCLUDE_NORMAL = False              # if True: treat "normal" as empty masks (no box)
IMG_SIZE = 1024                     # SAM pipeline expects 1024 (processor handles resize)
BATCH_SIZE = 1                      # memory-friendly; keep effective batch via GRAD_ACC
GRAD_ACC = 4                        # effective batch = BATCH_SIZE * GRAD_ACC
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-2
WORKERS = 2
SEED = 42
AMP = True
AMP_DTYPE = "float16"             # "bfloat16" recommended on A100/RTX30+; can switch to "float16" if needed
TINY_Q = 0.30                       # bottom-30% lesions by area are "tiny"
TINY_BOOST = 2.0                    # loss multiplier for tiny lesions
PATIENCE = 10                       # early stopping patience (epochs)
MODEL_ID = "wanglab/medsam-vit-base"  # MedSAM (ViT-B) checkpoint (HF Hub)

# ---------- SETUP ----------
import random, warnings, math, json
from pathlib import Path
from typing import List, Tuple, Optional

warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from tqdm.auto import tqdm
from PIL import Image
import albumentations as A

from transformers import SamModel, SamProcessor, get_cosine_schedule_with_warmup

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
set_seed(SEED)

# ---------- DATA ACQUISITION ----------
def ensure_busi_data(data_dir: Optional[str]) -> str:
    if data_dir is None:
        print("DATA_DIR is None -> downloading BUSI with kagglehub…")
        import kagglehub
        path = kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset")
        print("kagglehub path:", path)
        return str(path)
    p = Path(data_dir).expanduser().resolve()
    if not p.exists():
        raise SystemExit(f"DATA_DIR does not exist: {p}")
    return str(p)

def auto_find_busi_root(base_dir: str) -> str:
    base = Path(base_dir)
    cands = list(base.rglob("Dataset_BUSI_with_GT")) + list(base.rglob("Breast Ultrasound Images Dataset"))
    for p in cands:
        if (p/"benign").exists() and (p/"malignant").exists():
            return str(p)
        if (p.parent/"benign").exists() and (p.parent/"malignant").exists():
            return str(p.parent)
    for p in base.rglob("benign"):
        if (p.parent/"malignant").exists():
            return str(p.parent)
    return ""

DATA_DIR = ensure_busi_data(DATA_DIR)
BUSI_BASE = auto_find_busi_root(DATA_DIR)
if not BUSI_BASE:
    raise SystemExit("Could not locate BUSI inside DATA_DIR.")
print("Using BUSI_ROOT:", BUSI_BASE)

candidate = Path(BUSI_BASE) / "Dataset_BUSI_with_GT"
DATA_BASE = candidate if candidate.exists() else Path(BUSI_BASE)
print("Resolved data base:", DATA_BASE)

# ---------- AUGS ----------
def build_train_tf():
    # Ultrasound-friendly: CLAHE, mild geometry, gain/contrast, light noise
    return A.Compose([
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8,8), p=0.20),
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(IMG_SIZE, IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=12,
                           border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0, p=0.60),
        A.RandomBrightnessContrast(0.10, 0.10, p=0.30),
        A.GaussNoise(var_limit=(5.0, 20.0), p=0.25),
    ])

def build_eval_tf():
    return A.Compose([
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(IMG_SIZE, IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0),
    ])

# ---------- DATASET ----------
class BUSISegForSAM(Dataset):
    """
    Produces:
      - RGB image (H,W,3) uint8
      - binary mask float32 {0,1} (H,W)
      - bounding box [x1,y1,x2,y2] from mask (or None if empty)
    Accepts:
      DATA_BASE/{benign,malignant,normal} or DATA_BASE/Dataset_BUSI_with_GT/{...}
    """
    def __init__(self, root: str, include_normal: bool = False, train: bool = False):
        self.root = Path(root)
        base = self.root if (self.root/"benign").exists() else (self.root/"Dataset_BUSI_with_GT")
        classes = ["benign", "malignant"] + (["normal"] if include_normal else [])
        exts = (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG")

        self.samples: List[Tuple[str, Optional[str]]] = []
        for c in classes:
            cdir = base / c
            if not cdir.exists(): continue
            for p in sorted(cdir.iterdir()):
                if not p.is_file(): continue
                if "_mask" in p.stem: continue
                if p.suffix not in exts: continue
                if c == "normal":
                    self.samples.append((str(p), None))
                else:
                    stem = p.stem
                    found = False
                    for ext in exts:
                        for suf in ["_mask", "_mask_1", "_mask_2", "_mask_3"]:
                            q = p.with_name(stem + suf + ext)
                            if q.exists():
                                self.samples.append((str(p), str(q)))
                                found = True; break
                        if found: break

        if len(self.samples) == 0:
            raise RuntimeError(f"No BUSI samples under {base}")

        self.tf = build_train_tf() if train else build_eval_tf()
        self.train_flag = train

        # Precompute mask area for tiny-lesion weighting (on raw masks)
        areas = []
        for img_path, mask_path in self.samples:
            if mask_path is None: areas.append(0.0); continue
            m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if m is None: areas.append(0.0); continue
            a = float((m > 0).sum()) / (m.shape[0] * m.shape[1])
            areas.append(a)
        self.mask_area_frac = np.array(areas, dtype=np.float32)

    def __len__(self): return len(self.samples)

    @staticmethod
    def _mask_to_box(mask: np.ndarray):
        ys, xs = np.where(mask > 0.5)
        if len(xs) == 0 or len(ys) == 0:
            return None
        x1, x2 = int(xs.min()), int(xs.max())
        y1, y2 = int(ys.min()), int(ys.max())
        # expand box slightly for robustness
        h, w = mask.shape
        pad_x = max(1, int(0.02*w)); pad_y = max(1, int(0.02*h))
        x1 = max(0, x1 - pad_x); y1 = max(0, y1 - pad_y)
        x2 = min(w-1, x2 + pad_x); y2 = min(h-1, y2 + pad_y)
        return [x1, y1, x2, y2]

    @staticmethod
    def _to_rgb(gray: np.ndarray) -> np.ndarray:
        if gray.ndim == 2:
            return np.stack([gray, gray, gray], axis=-1)
        if gray.shape[-1] == 1:
            return np.repeat(gray, 3, axis=-1)
        return gray

    def __getitem__(self, i: int):
        img_path, mask_path = self.samples[i]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None: raise RuntimeError(f"Failed to read image {img_path}")

        if mask_path is None:
            mask = np.zeros_like(img, dtype=np.uint8)
        else:
            # merge multiple masks if present
            stem = Path(img_path).stem
            exts = (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG")
            merged = None
            for ext in exts:
                for suf in ["_mask", "_mask_1", "_mask_2", "_mask_3"]:
                    q = Path(img_path).with_name(stem + suf + ext)
                    if q.exists():
                        m = cv2.imread(str(q), cv2.IMREAD_GRAYSCALE)
                        if m is None: continue
                        m = (m > 0).astype(np.uint8)
                        merged = m if merged is None else np.maximum(merged, m)
            mask = merged if merged is not None else np.zeros_like(img, dtype=np.uint8)

        # Albumentations expects HWC
        aug = self.tf(image=img, mask=mask)
        g = aug["image"]      # uint8
        m = aug["mask"].astype(np.float32)

        # SAM expects RGB uint8, PIL/np ok
        rgb = self._to_rgb(g)

        # Compute bounding box from *augmented* mask
        box = self._mask_to_box(m)

        # tiny-lesion flag (based on original precomputed area fraction)
        tiny = self.mask_area_frac[i] <= np.quantile(self.mask_area_frac, TINY_Q) if self.train_flag else False

        return rgb, m, (box if box is not None else None), tiny, img_path

# Custom collate: return lists (processor will batch later)
def sam_collate(batch):
    imgs, masks, boxes, tiny_flags, paths = [], [], [], [], []
    for rgb, m, box, tiny, p in batch:
        imgs.append(rgb)
        masks.append(m)
        boxes.append(None if box is None else [box])  # list-of-one box per image
        tiny_flags.append(tiny)
        paths.append(p)
    return imgs, masks, boxes, tiny_flags, paths

# ---------- LOSSES & METRICS ----------
class DiceLoss(nn.Module):
    def __init__(self, eps=1e-7): super().__init__(); self.eps = eps
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num = 2*(probs*targets).sum((2,3)) + self.eps
        den = probs.sum((2,3)) + targets.sum((2,3)) + self.eps
        return 1 - (num/den).mean()

@torch.no_grad()
def dice_from_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    num = 2*(probs*targets).sum((2,3)) + eps
    den = probs.sum((2,3)) + targets.sum((2,3)) + eps
    return (num/den).mean().item()

@torch.no_grad()
def iou_from_probs(probs, targets, eps=1e-7, thr=0.5):
    preds = (probs > thr).float()
    inter = (preds*targets).sum((2,3))
    union = (preds + targets - preds*targets).sum((2,3))
    return ((inter+eps)/(union+eps)).mean().item()

@torch.no_grad()
def f1_from_probs(probs, targets, eps=1e-7, thr=0.5):
    preds = (probs > thr).float()
    tp = (preds*targets).sum((2,3))
    fp = (preds*(1-targets)).sum((2,3))
    fn = ((1-preds)*targets).sum((2,3))
    prec = (tp+eps)/(tp+fp+eps)
    rec  = (tp+eps)/(tp+fn+eps)
    return (2*prec*rec/(prec+rec+eps)).mean().item()

# ---------- BUILD LOADERS ----------
base_list_ds = BUSISegForSAM(str(DATA_BASE), include_normal=INCLUDE_NORMAL, train=False)
n_total = len(base_list_ds)
n_tr = int(0.8*n_total); n_va = int(0.1*n_total); n_te = n_total - n_tr - n_va
print(f"Total={n_total} -> train={n_tr}, val={n_va}, test={n_te}")

g = torch.Generator().manual_seed(SEED)
tr_idx, va_idx, te_idx = random_split(base_list_ds, [n_tr, n_va, n_te], generator=g)

train_full = BUSISegForSAM(str(DATA_BASE), include_normal=INCLUDE_NORMAL, train=True)
val_full   = BUSISegForSAM(str(DATA_BASE), include_normal=INCLUDE_NORMAL, train=False)
test_full  = BUSISegForSAM(str(DATA_BASE), include_normal=INCLUDE_NORMAL, train=False)

train_ds = Subset(train_full, tr_idx.indices)
val_ds   = Subset(val_full,   va_idx.indices)
test_ds  = Subset(test_full,  te_idx.indices)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=WORKERS, pin_memory=True, collate_fn=sam_collate)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=WORKERS, pin_memory=True, collate_fn=sam_collate)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=WORKERS, pin_memory=True, collate_fn=sam_collate)

# ---------- MODEL ----------
print("Loading MedSAM:", MODEL_ID)
processor = SamProcessor.from_pretrained(MODEL_ID)

# Load model in BF16 for memory; enable grad checkpointing
dtype_map = {"bfloat16": torch.bfloat16, "float16": torch.float16}
model = SamModel.from_pretrained(MODEL_ID)
model.gradient_checkpointing_enable()
model.config.use_cache = False
model.to(device)

# Defensive patch for old/new builds where SamAttention might lack dropout_p
for m in model.modules():
    if m.__class__.__name__ == "SamAttention" and not hasattr(m, "dropout_p"):
        m.dropout_p = 0.0

# Freeze prompt encoder; fine-tune vision encoder + mask decoder
for p in model.prompt_encoder.parameters():
    p.requires_grad = False

# ---------- OPTIM ----------
bce = nn.BCEWithLogitsLoss()
dice = DiceLoss()

train_steps_per_epoch = max(1, math.ceil(len(train_loader) / max(1, GRAD_ACC)))
num_train_steps = train_steps_per_epoch * EPOCHS

opt = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                        lr=LR, weight_decay=WEIGHT_DECAY)
sched = get_cosine_schedule_with_warmup(
    opt,
    num_warmup_steps=max(10, train_steps_per_epoch // 2),
    num_training_steps=num_train_steps
)

scaler = torch.amp.GradScaler(device="cuda", enabled=(AMP and device.type=="cuda"))
amp_dtype_torch = dtype_map.get(AMP_DTYPE, torch.bfloat16)

# ---------- TRAIN/EVAL ----------
os.makedirs(RUNS_DIR, exist_ok=True)
best_dice, best_path = -1.0, os.path.join(RUNS_DIR, "best_medsam.pt")
pat_bad = 0

def _batch_to_inputs(images, boxes):
    """
    images: list of HxWx3 uint8   boxes: list of [[x1,y1,x2,y2]] or None
    We skip samples with no box (i.e., empty masks when INCLUDE_NORMAL=True).
    """
    keep_imgs, keep_boxes, keep_idx = [], [], []
    for idx, (im, bx) in enumerate(zip(images, boxes)):
        if bx is None:
            continue
        keep_imgs.append(Image.fromarray(im))
        keep_boxes.append(bx)  # [[x1,y1,x2,y2]]
        keep_idx.append(idx)
    return keep_imgs, keep_boxes, keep_idx

def train_or_eval(loader, train=True):
    global best_dice, pat_bad

    mode = "train" if train else "eval"
    model.train(mode == "train")
    if train:
        opt.zero_grad(set_to_none=True)
    acc_count = 0

    total, run_loss, d_log, i_log, f1_log = 0, 0.0, 0.0, 0.0, 0.0
    pbar = tqdm(loader, desc=f"[{mode}]")

    for images, masks, boxes, tiny_flags, _ in pbar:
        keep_imgs, keep_boxes, keep_idx = _batch_to_inputs(images, boxes)
        if len(keep_imgs) == 0:
            continue

        sel_masks = [masks[i] for i in keep_idx]
        sel_tiny  = [tiny_flags[i] for i in keep_idx]

        # Processor: resize to 1024, normalize, build box prompts
        inputs = processor(images=keep_imgs, input_boxes=keep_boxes, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)  # (B,3,1024,1024)
        input_boxes  = inputs["input_boxes"].to(device)   # (B,1,4)

        # Downsample GT masks to SAM low-res (256x256) for differentiable loss
        gt = []
        for m in sel_masks:
            m_t = torch.from_numpy(m).float().unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
            m_ds = F.interpolate(m_t, size=(256,256), mode="nearest")
            gt.append(m_ds)
        gt = torch.cat(gt, dim=0).to(device)  # (B,1,256,256)

        # Forward
        with torch.amp.autocast(device_type="cuda", dtype=amp_dtype_torch, enabled=(AMP and device.type=="cuda")):
            out = model(pixel_values=pixel_values, input_boxes=input_boxes, multimask_output=False)
            # out.pred_masks: (B, 1, 1, 256, 256)  -> select first box & first mask
            logits = out.pred_masks[:, 0, 0, :, :].unsqueeze(1)  # (B,1,256,256)

            ce = bce(logits, gt)
            dl = dice(logits, gt)
            base_loss = ce + dl

            if any(sel_tiny):
                weights = torch.tensor([TINY_BOOST if t else 1.0 for t in sel_tiny],
                                       device=device).view(-1,1,1,1)
                loss = (base_loss * weights).mean()
            else:
                loss = base_loss

            # gradient accumulation
            loss = loss / max(1, GRAD_ACC)

        if train:
            scaler.scale(loss).backward()
            acc_count += 1
            if (acc_count % GRAD_ACC) == 0:
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()

        # On-the-fly metrics @thr=0.5 on low-res
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            d_log += dice_from_logits(logits, gt) * pixel_values.size(0)
            i_log += iou_from_probs(probs, gt, thr=0.5) * pixel_values.size(0)
            f1_log += f1_from_probs(probs, gt, thr=0.5) * pixel_values.size(0)
        run_loss += loss.item() * pixel_values.size(0)
        total += pixel_values.size(0)

        pbar.set_postfix(loss=f"{loss.item():.4f}",
                         dice=f"{(d_log/total):.4f}",
                         iou=f"{(i_log/total):.4f}")

    # Final optimizer step if an epoch ended mid-accumulation
    if train and (acc_count % max(1, GRAD_ACC) != 0):
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        sched.step()

    if total == 0:
        return float("inf"), 0.0, 0.0, 0.0
    return run_loss/total, d_log/total, i_log/total, f1_log/total

print("🚀 Training MedSAM (ViT-B) on BUSI with box prompts")
best_state = None
for ep in range(1, EPOCHS+1):
    tr_loss, tr_d, tr_i, tr_f1 = train_or_eval(train_loader, train=True)
    va_loss, va_d, va_i, va_f1 = train_or_eval(val_loader,   train=False)
    print(f"Epoch {ep:03d} | "
          f"train: loss={tr_loss:.4f} dice={tr_d:.4f} iou={tr_i:.4f} f1={tr_f1:.4f}  ||  "
          f"val: loss={va_loss:.4f} dice={va_d:.4f} iou={va_i:.4f} f1={va_f1:.4f}")

    if va_d > best_dice:
        best_dice = va_d; pat_bad = 0
        best_state = {
            "model": model.state_dict(),
            "epoch": ep,
            "val_dice": va_d,
            "val_iou": va_i,
            "val_f1": va_f1,
            "processor": MODEL_ID
        }
        torch.save(best_state, best_path)
        print(f"  ✓ Saved new best (Dice={va_d:.4f}) → {best_path}")
    else:
        pat_bad += 1
        if pat_bad >= PATIENCE:
            print(f"Early stopping at epoch {ep} (no val Dice improvement for {PATIENCE} epochs)")
            break

# ---------- TEST ----------
if best_state is None:
    best_state = torch.load(best_path, map_location=device)
model.load_state_dict(best_state["model"])

@torch.no_grad()
def eval_lowres(loader, thr=0.5):
    model.eval()
    total=0; d=i=f1=0.0
    for images, masks, boxes, tiny_flags, _ in tqdm(loader, desc="[test]"):
        keep_imgs, keep_boxes, keep_idx = _batch_to_inputs(images, boxes)
        if len(keep_imgs) == 0:
            continue

        inputs = processor(images=keep_imgs, input_boxes=keep_boxes, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)
        input_boxes  = inputs["input_boxes"].to(device)

        with torch.amp.autocast(device_type="cuda", dtype=amp_dtype_torch, enabled=(AMP and device.type=="cuda")):
            out = model(pixel_values=pixel_values, input_boxes=input_boxes, multimask_output=False)
            logits = out.pred_masks[:, 0, 0, :, :].unsqueeze(1)  # (B,1,256,256)
            probs_low = torch.sigmoid(logits)

        sel_masks = [masks[i] for i in keep_idx]
        gt = []
        for m in sel_masks:
            m_t = torch.from_numpy(m).float().unsqueeze(0).unsqueeze(0)
            m_ds = F.interpolate(m_t, size=(256,256), mode="nearest")
            gt.append(m_ds)
        gt = torch.cat(gt, dim=0).to(device)  # (B,1,256,256)

        d  += dice_from_logits(logits, gt) * pixel_values.size(0)
        i  += iou_from_probs(probs_low, gt, thr=thr) * pixel_values.size(0)
        f1 += f1_from_probs(probs_low, gt, thr=thr) * pixel_values.size(0)
        total += pixel_values.size(0)
    return d/total, i/total, f1/total

td, ti, tf1 = eval_lowres(test_loader, thr=0.5)
print(f"TEST (low-res metrics) → Dice={td:.4f} | IoU={ti:.4f} | F1={tf1:.4f}")
print("Best checkpoint:", best_path)

# ---------- SAVE HF-FRIENDLY STATE_DICT ----------
SAVE_DIR = os.path.join(RUNS_DIR, "hf_medsam_finetuned")
os.makedirs(SAVE_DIR, exist_ok=True)
with open(os.path.join(SAVE_DIR, "processor_id.txt"), "w") as f:
    f.write(MODEL_ID + "\n")
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "pytorch_model.bin"))
print(f"Saved state_dict to {SAVE_DIR}")


Device: cuda
DATA_DIR is None -> downloading BUSI with kagglehub…
kagglehub path: /kaggle/input/breast-ultrasound-images-dataset
Using BUSI_ROOT: /kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT
Resolved data base: /kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT
Total=647 -> train=517, val=64, test=66
Loading MedSAM: wanglab/medsam-vit-base
🚀 Training MedSAM (ViT-B) on BUSI with box prompts


[train]:   0%|          | 0/517 [00:00<?, ?it/s]

[eval]:   0%|          | 0/64 [00:00<?, ?it/s]

Epoch 001 | train: loss=0.0618 dice=0.8620 iou=0.7843 f1=0.8741  ||  val: loss=0.0520 dice=0.8543 iou=0.7743 f1=0.8661
  ✓ Saved new best (Dice=0.8543) → /content/runs_busi_medsam_box/best_medsam.pt


[train]:   0%|          | 0/517 [00:00<?, ?it/s]

...

[train]:   0%|          | 0/517 [00:00<?, ?it/s]

[eval]:   0%|          | 0/64 [00:00<?, ?it/s]

Epoch 056 | train: loss=0.0163 dice=0.9606 iou=0.9323 f1=0.9647  ||  val: loss=0.0284 dice=0.9246 iou=0.8684 f1=0.9278
Early stopping at epoch 56 (no val Dice improvement for 10 epochs)


[test]:   0%|          | 0/66 [00:00<?, ?it/s]

TEST (low-res metrics) → Dice=0.9339 | IoU=0.8825 | F1=0.9362
Best checkpoint: /content/runs_busi_medsam_box/best_medsam.pt
Saved state_dict to /content/runs_busi_medsam_box/hf_medsam_finetuned


In [6]:
# %% [colab] BUSI → MedSAM fine-tune + IoU-oriented validation + TTA + multi-mask selection
# =================================================================================================

# ---- ENV/MEM ----
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

# ---- INSTALL ----
!pip -q install --no-input "transformers==4.53.3" "accelerate>=0.33" kagglehub albumentations opencv-python pillow timm==0.9.16

# ---- CONFIG ----
DATA_DIR = None
RUNS_DIR = "/content/runs_busi_medsam_iou"
INCLUDE_NORMAL = False
IMG_SIZE = 1024

# training
BATCH_SIZE = 1
GRAD_ACC = 4
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-2
SEED = 42
WORKERS = 2
AMP = True                       # FP16 autocast + GradScaler
TINY_Q = 0.30
TINY_BOOST = 2.0
PATIENCE = 10
MODEL_ID = "wanglab/medsam-vit-base"

# loss mix (keep BCE+Dice; add Soft-Jaccard for IoU)
W_BCE, W_DICE, W_JACCARD = 0.5, 0.5, 0.2

# validation / inference
THR_SWEEP = (0.30, 0.80, 21)     # (min, max, steps) — pick thr that maximizes IoU on VAL
TTA_SCALES = (0.75, 1.00, 1.25, 1.50)
TTA_HFLIP = True
MULTIMASK = True                 # request 3 candidate masks and pick the best by predicted IoU

# optional, very light morphology after binarization (post)
APPLY_MORPH = True
MORPH_OPEN_KERNEL = 3            # 3x3 opening to remove specks
MORPH_CLOSE_KERNEL = 3           # 3x3 closing to fill pinholes

# ---- SETUP ----
import random, warnings, math
from pathlib import Path
from typing import List, Tuple, Optional
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from tqdm.auto import tqdm
from PIL import Image
import albumentations as A
from transformers import SamModel, SamProcessor, get_cosine_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
torch.backends.cuda.matmul.allow_tf32 = True

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
set_seed(SEED)

# ---- DATA ----
def ensure_busi_data(data_dir: Optional[str]) -> str:
    if data_dir is None:
        print("DATA_DIR is None -> downloading BUSI with kagglehub…")
        import kagglehub
        path = kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset")
        print("kagglehub path:", path)
        return str(path)
    p = Path(data_dir).expanduser().resolve()
    if not p.exists():
        raise SystemExit(f"DATA_DIR does not exist: {p}")
    return str(p)

def auto_find_busi_root(base_dir: str) -> str:
    base = Path(base_dir)
    cands = list(base.rglob("Dataset_BUSI_with_GT")) + list(base.rglob("Breast Ultrasound Images Dataset"))
    for p in cands:
        if (p/"benign").exists() and (p/"malignant").exists():
            return str(p)
        if (p.parent/"benign").exists() and (p.parent/"malignant").exists():
            return str(p.parent)
    for p in base.rglob("benign"):
        if (p.parent/"malignant").exists():
            return str(p.parent)
    return ""

DATA_DIR = ensure_busi_data(DATA_DIR)
BUSI_BASE = auto_find_busi_root(DATA_DIR)
if not BUSI_BASE:
    raise SystemExit("Could not locate BUSI inside DATA_DIR.")
print("Using BUSI_ROOT:", BUSI_BASE)
candidate = Path(BUSI_BASE) / "Dataset_BUSI_with_GT"
DATA_BASE = candidate if candidate.exists() else Path(BUSI_BASE)
print("Resolved data base:", DATA_BASE)

def build_train_tf():
    return A.Compose([
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8,8), p=0.20),
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(IMG_SIZE, IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=12,
                           border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0, p=0.60),
        A.RandomBrightnessContrast(0.10, 0.10, p=0.30),
        A.GaussNoise(var_limit=(5.0, 20.0), p=0.25),
    ])

def build_eval_tf():
    return A.Compose([
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(IMG_SIZE, IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0),
    ])

class BUSISegForSAM(Dataset):
    def __init__(self, root: str, include_normal: bool = False, train: bool = False):
        self.root = Path(root)
        base = self.root if (self.root/"benign").exists() else (self.root/"Dataset_BUSI_with_GT")
        classes = ["benign", "malignant"] + (["normal"] if include_normal else [])
        exts = (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG")
        self.samples: List[Tuple[str, Optional[str]]] = []
        for c in classes:
            cdir = base / c
            if not cdir.exists(): continue
            for p in sorted(cdir.iterdir()):
                if not p.is_file(): continue
                if "_mask" in p.stem: continue
                if p.suffix not in exts: continue
                if c == "normal":
                    self.samples.append((str(p), None))
                else:
                    stem = p.stem
                    found = False
                    for ext in exts:
                        for suf in ["_mask", "_mask_1", "_mask_2", "_mask_3"]:
                            q = p.with_name(stem + suf + ext)
                            if q.exists():
                                self.samples.append((str(p), str(q))); found = True; break
                        if found: break
        if len(self.samples) == 0:
            raise RuntimeError(f"No BUSI samples under {base}")
        self.tf = build_train_tf() if train else build_eval_tf()
        self.train_flag = train
        areas = []
        for _, mask_path in self.samples:
            if mask_path is None: areas.append(0.0); continue
            m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if m is None: areas.append(0.0); continue
            a = float((m > 0).sum()) / (m.shape[0] * m.shape[1])
            areas.append(a)
        self.mask_area_frac = np.array(areas, dtype=np.float32)

    def __len__(self): return len(self.samples)

    @staticmethod
    def _mask_to_box(mask: np.ndarray):
        ys, xs = np.where(mask > 0.5)
        if len(xs) == 0 or len(ys) == 0:
            return None
        x1, x2 = int(xs.min()), int(xs.max())
        y1, y2 = int(ys.min()), int(ys.max())
        h, w = mask.shape
        pad_x = max(1, int(0.02*w)); pad_y = max(1, int(0.02*h))
        return [max(0, x1 - pad_x), max(0, y1 - pad_y),
                min(w-1, x2 + pad_x), min(h-1, y2 + pad_y)]

    @staticmethod
    def _to_rgb(gray: np.ndarray) -> np.ndarray:
        if gray.ndim == 2: return np.stack([gray, gray, gray], axis=-1)
        if gray.shape[-1] == 1: return np.repeat(gray, 3, axis=-1)
        return gray

    def __getitem__(self, i: int):
        img_path, mask_path = self.samples[i]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None: raise RuntimeError(f"Failed to read image {img_path}")
        merged = None
        if mask_path is not None:
            stem = Path(img_path).stem
            exts = (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG")
            for ext in exts:
                for suf in ["_mask", "_mask_1", "_mask_2", "_mask_3"]:
                    q = Path(img_path).with_name(stem + suf + ext)
                    if q.exists():
                        m = cv2.imread(str(q), cv2.IMREAD_GRAYSCALE)
                        if m is None: continue
                        m = (m > 0).astype(np.uint8)
                        merged = m if merged is None else np.maximum(merged, m)
        mask = merged if merged is not None else np.zeros_like(img, dtype=np.uint8)
        aug = self.tf(image=img, mask=mask)
        g = aug["image"]
        m = aug["mask"].astype(np.float32)
        rgb = self._to_rgb(g)
        box = self._mask_to_box(m)
        tiny = self.mask_area_frac[i] <= np.quantile(self.mask_area_frac, TINY_Q) if self.train_flag else False
        return rgb, m, (box if box is not None else None), tiny, img_path

def sam_collate(batch):
    imgs, masks, boxes, tiny_flags, paths = [], [], [], [], []
    for rgb, m, box, tiny, p in batch:
        imgs.append(rgb); masks.append(m)
        boxes.append(None if box is None else [box])
        tiny_flags.append(tiny); paths.append(p)
    return imgs, masks, boxes, tiny_flags, paths

# ---- LOSSES/METRICS ----
class DiceLoss(nn.Module):
    def __init__(self, eps=1e-7): super().__init__(); self.eps = eps
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num = 2*(probs*targets).sum((2,3)) + self.eps
        den = probs.sum((2,3)) + targets.sum((2,3)) + self.eps
        return 1 - (num/den).mean()

class SoftJaccardLoss(nn.Module):
    def __init__(self, eps=1e-7): super().__init__(); self.eps = eps
    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        inter = (p*targets).sum((2,3))
        union = p.sum((2,3)) + targets.sum((2,3)) - inter
        j = (inter + self.eps) / (union + self.eps)
        return (1 - j).mean()

@torch.no_grad()
def dice_from_logits(logits, targets, eps=1e-7):
    p = torch.sigmoid(logits)
    num = 2*(p*targets).sum((2,3)) + eps
    den = p.sum((2,3)) + targets.sum((2,3)) + eps
    return (num/den).mean().item()

@torch.no_grad()
def iou_from_probs(probs, targets, eps=1e-7, thr=0.5):
    preds = (probs > thr).float()
    inter = (preds*targets).sum((2,3))
    union = (preds + targets - preds*targets).sum((2,3))
    return ((inter+eps)/(union+eps)).mean().item()

@torch.no_grad()
def f1_from_probs(probs, targets, eps=1e-7, thr=0.5):
    preds = (probs > thr).float()
    tp = (preds*targets).sum((2,3))
    fp = (preds*(1-targets)).sum((2,3))
    fn = ((1-preds)*targets).sum((2,3))
    prec = (tp+eps)/(tp+fp+eps)
    rec  = (tp+eps)/(tp+fn+eps)
    return (2*prec*rec/(prec+rec+eps)).mean().item()

# ---- LOADERS ----
base_list_ds = BUSISegForSAM(str(DATA_BASE), include_normal=INCLUDE_NORMAL, train=False)
n_total = len(base_list_ds)
n_tr = int(0.8*n_total); n_va = int(0.1*n_total); n_te = n_total - n_tr - n_va
print(f"Total={n_total} -> train={n_tr}, val={n_va}, test={n_te}")

g = torch.Generator().manual_seed(SEED)
tr_idx, va_idx, te_idx = random_split(base_list_ds, [n_tr, n_va, n_te], generator=g)

train_full = BUSISegForSAM(str(DATA_BASE), include_normal=INCLUDE_NORMAL, train=True)
val_full   = BUSISegForSAM(str(DATA_BASE), include_normal=INCLUDE_NORMAL, train=False)
test_full  = BUSISegForSAM(str(DATA_BASE), include_normal=INCLUDE_NORMAL, train=False)

train_ds = Subset(train_full, tr_idx.indices)
val_ds   = Subset(val_full,   va_idx.indices)
test_ds  = Subset(test_full,  te_idx.indices)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=WORKERS, pin_memory=True, collate_fn=sam_collate)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=WORKERS, pin_memory=True, collate_fn=sam_collate)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=WORKERS, pin_memory=True, collate_fn=sam_collate)

# ---- MODEL ----
print("Loading MedSAM:", MODEL_ID)
processor = SamProcessor.from_pretrained(MODEL_ID)
model = SamModel.from_pretrained(MODEL_ID)   # FP32 weights for stability with GradScaler
model.gradient_checkpointing_enable()
model.config.use_cache = False
model.to(device)

# safety patch for older/newer builds
for m in model.modules():
    if m.__class__.__name__ == "SamAttention" and not hasattr(m, "dropout_p"):
        m.dropout_p = 0.0

# freeze prompt encoder
for p in model.prompt_encoder.parameters():
    p.requires_grad = False

# ---- OPTIM ----
bce = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()
jaccard = SoftJaccardLoss()

opt = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                        lr=LR, weight_decay=WEIGHT_DECAY)

train_steps_per_epoch = max(1, math.ceil(len(train_loader) / max(1, GRAD_ACC)))
num_train_steps = train_steps_per_epoch * EPOCHS
sched = get_cosine_schedule_with_warmup(
    opt,
    num_warmup_steps=max(10, train_steps_per_epoch // 2),
    num_training_steps=num_train_steps
)

scaler = torch.amp.GradScaler(device="cuda", enabled=(AMP and device.type=="cuda"))

# ---- TRAIN/EVAL (single-mask during training for memory) ----
os.makedirs(RUNS_DIR, exist_ok=True)
best_dice, best_path = -1.0, os.path.join(RUNS_DIR, "best_medsam.pt")
pat_bad = 0

def _batch_to_inputs(images, boxes):
    keep_imgs, keep_boxes, keep_idx = [], [], []
    for idx, (im, bx) in enumerate(zip(images, boxes)):
        if bx is None:  # skip empties
            continue
        keep_imgs.append(Image.fromarray(im))
        keep_boxes.append(bx)
        keep_idx.append(idx)
    return keep_imgs, keep_boxes, keep_idx

def train_or_eval(loader, train=True):
    model.train(train)
    if train: opt.zero_grad(set_to_none=True)
    acc = 0; total=0; run_loss=d_log=i_log=f1_log=0.0
    pbar = tqdm(loader, desc="[train]" if train else "[eval]")

    for images, masks, boxes, tiny_flags, _ in pbar:
        keep_imgs, keep_boxes, keep_idx = _batch_to_inputs(images, boxes)
        if len(keep_imgs) == 0: continue

        sel_masks = [masks[i] for i in keep_idx]
        sel_tiny  = [tiny_flags[i] for i in keep_idx]

        inputs = processor(images=keep_imgs, input_boxes=keep_boxes, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)  # (B,3,1024,1024)
        input_boxes  = inputs["input_boxes"].to(device)   # (B,1,4)

        gt = []
        for m in sel_masks:
            m_t = torch.from_numpy(m).float().unsqueeze(0).unsqueeze(0)
            m_ds = F.interpolate(m_t, size=(256,256), mode="nearest")
            gt.append(m_ds)
        gt = torch.cat(gt, dim=0).to(device)  # (B,1,256,256)

        with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=(AMP and device.type=="cuda")):
            out = model(pixel_values=pixel_values, input_boxes=input_boxes, multimask_output=False)
            logits = out.pred_masks[:, 0, 0, :, :].unsqueeze(1)  # (B,1,256,256)

            ce = bce(logits, gt)
            dl = dice_loss(logits, gt)
            jl = jaccard(logits, gt)

            base_loss = W_BCE*ce + W_DICE*dl + W_JACCARD*jl
            if any(sel_tiny):
                weights = torch.tensor([TINY_BOOST if t else 1.0 for t in sel_tiny],
                                       device=device).view(-1,1,1,1)
                loss = (base_loss * weights).mean()
            else:
                loss = base_loss

            loss = loss / max(1, GRAD_ACC)

        if train:
            scaler.scale(loss).backward()
            acc += 1
            if (acc % GRAD_ACC) == 0:
                scaler.step(opt); scaler.update()
                opt.zero_grad(set_to_none=True); sched.step()

        with torch.no_grad():
            probs = torch.sigmoid(logits)
            d_log += dice_from_logits(logits, gt) * pixel_values.size(0)
            i_log += iou_from_probs(probs, gt, thr=0.5) * pixel_values.size(0)
            f1_log += f1_from_probs(probs, gt, thr=0.5) * pixel_values.size(0)
        run_loss += loss.item() * pixel_values.size(0)
        total += pixel_values.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}",
                         dice=f"{(d_log/total):.4f}",
                         iou=f"{(i_log/total):.4f}")

    if train and (acc % max(1, GRAD_ACC) != 0):
        scaler.step(opt); scaler.update()
        opt.zero_grad(set_to_none=True); sched.step()

    if total == 0: return float("inf"), 0.0, 0.0, 0.0
    return run_loss/total, d_log/total, i_log/total, f1_log/total

print("🚀 Training MedSAM (ViT-B) on BUSI with IoU-oriented pipeline")
best_state = None
for ep in range(1, EPOCHS+1):
    tr_loss, tr_d, tr_i, tr_f1 = train_or_eval(train_loader, train=True)
    va_loss, va_d, va_i, va_f1 = train_or_eval(val_loader,   train=False)
    print(f"Epoch {ep:03d} | train: loss={tr_loss:.4f} dice={tr_d:.4f} iou={tr_i:.4f} f1={tr_f1:.4f}  ||  "
          f"val: loss={va_loss:.4f} dice={va_d:.4f} iou={va_i:.4f} f1={va_f1:.4f}")

    if va_d > best_dice:
        best_dice = va_d; pat_bad = 0
        best_state = {"model": model.state_dict(), "epoch": ep,
                      "val_dice": va_d, "val_iou": va_i, "val_f1": va_f1, "processor": MODEL_ID}
        torch.save(best_state, os.path.join(RUNS_DIR, "best_medsam.pt"))
        print(f"  ✓ Saved new best (Dice={va_d:.4f})")
    else:
        pat_bad += 1
        if pat_bad >= PATIENCE:
            print(f"Early stopping at epoch {ep} (no val Dice improvement for {PATIENCE} epochs)")
            break

# ---- RELOAD BEST ----
ck = torch.load(os.path.join(RUNS_DIR, "best_medsam.pt"), map_location=device)
model.load_state_dict(ck["model"])

# ---- INFERENCE HELPERS: TTA + multi-mask choose-best ----
def _resize_np(img: np.ndarray, scale: float) -> np.ndarray:
    h, w = img.shape[:2]
    return np.array(Image.fromarray(img).resize((int(round(w*scale)), int(round(h*scale))), Image.BILINEAR))

def _flip_box(box, w):
    x1,y1,x2,y2 = box
    return [w-1-x2, y1, w-1-x1, y2]

@torch.no_grad()
def sam_forward_best_mask(images_pil, boxes_list, multimask=True):
    """
    images_pil: list of PIL RGB
    boxes_list: like [[[x1,y1,x2,y2]], ...]
    returns logits_best: (B,1,256,256), where best is argmax over predicted IoU (if multimask)
    """
    inputs = processor(images=images_pil, input_boxes=boxes_list, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)
    input_boxes  = inputs["input_boxes"].to(device)

    out = model(pixel_values=pixel_values, input_boxes=input_boxes, multimask_output=multimask)
    # pred_masks: (B, 1, K, 256, 256), iou_scores: (B, K)
    K = out.pred_masks.shape[2]
    if K == 1:
        logits = out.pred_masks[:, 0, 0, :, :].unsqueeze(1)  # (B,1,256,256)
        return logits

    scores = out.iou_scores            # (B, 1, K)
    idx = scores[:, 0, :].argmax(dim=-1)         # (B,)
    # gather best along K
    b = logits = out.pred_masks[:, 0, :, :, :]  # (B,K,256,256)
    gather = idx.view(-1,1,1,1).expand(-1,1,256,256)
    best = b.gather(1, gather).squeeze(1).unsqueeze(1)  # (B,1,256,256)
    return best

@torch.no_grad()
def tta_predict_prob(images_np, boxes_per_img, scales=TTA_SCALES, hflip=TTA_HFLIP, multimask=True):
    """
    images_np: list of RGB uint8 arrays (H,W,3)
    boxes_per_img: list of [[x1,y1,x2,y2]] (one per image)
    returns probs averaged over TTA: (B,1,256,256)
    """
    assert len(images_np) == len(boxes_per_img)
    B = len(images_np)
    prob_sum = None; count = 0

    for s in scales:
        # scale images and boxes
        imgs_scaled, boxes_scaled = [], []
        for img, boxes in zip(images_np, boxes_per_img):
            H, W = img.shape[:2]
            img_s = _resize_np(img, s)
            sx = img_s.shape[1] / W; sy = img_s.shape[0] / H
            adj = [[int(round(b[0]*sx)), int(round(b[1]*sy)),
                    int(round(b[2]*sx)), int(round(b[3]*sy))] for b in boxes]
            imgs_scaled.append(Image.fromarray(img_s))
            boxes_scaled.append(adj)

        # original orientation
        logits = sam_forward_best_mask(imgs_scaled, boxes_scaled, multimask=multimask)  # (B,1,256,256)
        probs = torch.sigmoid(logits)
        prob_sum = probs if prob_sum is None else (prob_sum + probs)
        count += 1

        # horizontal flip
        if hflip:
            imgs_flip, boxes_flip = [], []
            for img, boxes in zip(images_np, boxes_per_img):
                img_s = _resize_np(img, s)
                W_s = img_s.shape[1]
                img_f = np.ascontiguousarray(img_s[:, ::-1, :])  # flip
                adj = [[*_flip_box(b, W_s)] for b in boxes]
                imgs_flip.append(Image.fromarray(img_f)); boxes_flip.append(adj)

            logits_f = sam_forward_best_mask(imgs_flip, boxes_flip, multimask=multimask)
            probs_f = torch.sigmoid(logits_f)
            # unflip the predicted masks before accumulation
            probs_f = torch.flip(probs_f, dims=[-1])
            prob_sum = prob_sum + probs_f
            count += 1

    return prob_sum / count

def _maybe_morph(bin_mask: np.ndarray) -> np.ndarray:
    if not APPLY_MORPH: return bin_mask
    k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (MORPH_OPEN_KERNEL, MORPH_OPEN_KERNEL))
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (MORPH_CLOSE_KERNEL, MORPH_CLOSE_KERNEL))
    m = cv2.morphologyEx(bin_mask, cv2.MORPH_OPEN, k_open)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k_close)
    return m

@torch.no_grad()
def eval_with_tta(loader, thr=0.5, use_val=False):
    model.eval()
    total=0; d=i=f1=0.0
    for images, masks, boxes, tiny_flags, _ in tqdm(loader, desc="[val-tta]" if use_val else "[test-tta]"):
        # collect valid samples with boxes
        imgs_np, boxes_in, keep_idx = [], [], []
        for idx, (im, bx) in enumerate(zip(images, boxes)):
            if bx is None: continue
            imgs_np.append(im)  # RGB np
            boxes_in.append(bx) # [[x1,y1,x2,y2]]
            keep_idx.append(idx)
        if len(imgs_np) == 0: continue

        probs = tta_predict_prob(imgs_np, boxes_in, scales=TTA_SCALES, hflip=TTA_HFLIP, multimask=MULTIMASK)  # (B,1,256,256)

        # binarize with thr and (optional) morphology — only for metrics
        bin_np = (probs.squeeze(1).cpu().numpy() > thr).astype(np.uint8)
        if APPLY_MORPH:
            bin_np = np.stack([_maybe_morph(m*255)//255 for m in bin_np], axis=0)

        # build GT at 256
        sel_masks = [masks[i] for i in keep_idx]
        gt = []
        for m in sel_masks:
            m_t = torch.from_numpy(m).float().unsqueeze(0).unsqueeze(0)
            m_ds = F.interpolate(m_t, size=(256,256), mode="nearest")
            gt.append(m_ds)
        gt = torch.cat(gt, dim=0).to(device)  # (B,1,256,256)

        # compute metrics using post-processed predictions
        preds = torch.from_numpy(bin_np).to(device).unsqueeze(1).float()
        inter = (preds*gt).sum((2,3))
        union = (preds + gt - preds*gt).sum((2,3))
        i_batch = ((inter+1e-7)/(union+1e-7)).mean().item()

        tp = (preds*gt).sum((2,3))
        fp = (preds*(1-gt)).sum((2,3))
        fn = ((1-preds)*gt).sum((2,3))
        dice_batch = ( (2*tp+1e-7)/( (2*tp)+fp+fn+1e-7) ).mean().item()
        prec = (tp+1e-7)/(tp+fp+1e-7)
        rec  = (tp+1e-7)/(tp+fn+1e-7)
        f1_batch = (2*prec*rec/(prec+rec+1e-7)).mean().item()

        d += dice_batch * len(imgs_np)
        i += i_batch * len(imgs_np)
        f1 += f1_batch * len(imgs_np)
        total += len(imgs_np)

    return (d/total, i/total, f1/total) if total>0 else (0.0,0.0,0.0)

@torch.no_grad()
def sweep_thr_for_iou(val_loader, thr_min=0.30, thr_max=0.80, steps=21):
    best_thr, best_iou, best_dice = 0.5, -1.0, 0.0
    for k in range(steps):
        thr = thr_min + k*( (thr_max-thr_min)/max(1,steps-1) )
        d,i,f1 = eval_with_tta(val_loader, thr=thr, use_val=True)
        print(f"  thr={thr:.3f} → VAL TTA: Dice={d:.4f} | IoU={i:.4f} | F1={f1:.4f}")
        if i > best_iou:
            best_iou, best_thr, best_dice = i, thr, d
    return best_thr, best_iou, best_dice

# ---- CHOOSE THRESHOLD ON VAL (IoU-optimal) ----
thr_star, val_iou_star, val_dice_at_star = sweep_thr_for_iou(
    val_loader, thr_min=THR_SWEEP[0], thr_max=THR_SWEEP[1], steps=THR_SWEEP[2]
)
print(f"Chosen threshold (IoU-optimal): thr*={thr_star:.3f} | VAL IoU={val_iou_star:.4f} | VAL Dice@thr*={val_dice_at_star:.4f}")

# ---- FINAL TEST with TTA + multi-mask + post ----
td, ti, tf1 = eval_with_tta(test_loader, thr=thr_star, use_val=False)
print(f"TEST (TTA + multi-mask + post) → Dice={td:.4f} | IoU={ti:.4f} | F1={tf1:.4f}")
print("Best checkpoint:", os.path.join(RUNS_DIR, "best_medsam.pt"))

# ---- SAVE STATE_DICT ----
SAVE_DIR = os.path.join(RUNS_DIR, "hf_medsam_finetuned")
os.makedirs(SAVE_DIR, exist_ok=True)
with open(os.path.join(SAVE_DIR, "processor_id.txt"), "w") as f:
    f.write(MODEL_ID + "\n")
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "pytorch_model.bin"))
print(f"Saved state_dict to {SAVE_DIR}")


Device: cuda
DATA_DIR is None -> downloading BUSI with kagglehub…
kagglehub path: /kaggle/input/breast-ultrasound-images-dataset
Using BUSI_ROOT: /kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT
Resolved data base: /kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT
Total=647 -> train=517, val=64, test=66
Loading MedSAM: wanglab/medsam-vit-base
🚀 Training MedSAM (ViT-B) on BUSI with IoU-oriented pipeline


[train]:   0%|          | 0/517 [00:00<?, ?it/s]

[eval]:   0%|          | 0/64 [00:00<?, ?it/s]

Epoch 001 | train: loss=0.0447 dice=0.8672 iou=0.7889 f1=0.8775  ||  val: loss=0.0387 dice=0.8567 iou=0.7686 f1=0.8641
  ✓ Saved new best (Dice=0.8567)


[train]:   0%|          | 0/517 [00:00<?, ?it/s]

...

  thr=0.775 → VAL TTA: Dice=0.1574 | IoU=0.1031 | F1=0.1574


[val-tta]:   0%|          | 0/64 [00:00<?, ?it/s]

  thr=0.800 → VAL TTA: Dice=0.1505 | IoU=0.0980 | F1=0.1505
Chosen threshold (IoU-optimal): thr*=0.375 | VAL IoU=0.8693 | VAL Dice@thr*=0.9285


[test-tta]:   0%|          | 0/66 [00:00<?, ?it/s]

TEST (TTA + multi-mask + post) → Dice=0.9332 | IoU=0.8772 | F1=0.9332
Best checkpoint: /content/runs_busi_medsam_iou/best_medsam.pt
Saved state_dict to /content/runs_busi_medsam_iou/hf_medsam_finetuned
